Load Solitude from OSM\n\nParses `data/osm/UT/Solitude.osm` into a `Mountain` object using the new backend (`core/support/mountain.py`). This drives the full ingestion pipeline: OSM parsing → elevation lookup → vertical/difficulty calculation → historical weather lookup.\n\nRun this notebook with the repo root as the working directory (it uses relative paths like `data/cache_db.db`)."

In [ ]:
from core.support.mountain import Mountain
from core.datamodels.season_pass import Season_Pass

OSM_FILE = "data/osm/UT/Solitude.osm"

Build the `Mountain` from the OSM file. This hits the real elevation and weather APIs (caching elevation lookups in `data/cache_db.db`), so it can take a bit on first run.

In [ ]:
season_passes = [Season_Pass.IKON]
url = "https://www.solitudemountain.com"

mountain = Mountain.from_osm(OSM_FILE, season_passes, url)
mountain

## Summary stats

In [ ]:
print(f"Name: {mountain.name}")
print(f"Mountain ID: {mountain.mountain_id}")
print(f"State: {mountain.state}")
print(f"Region: {mountain.region()}")
print(f"Coordinates: {mountain.coordinates}")
print(f"Direction: {mountain.direction}")
print(f"Vertical: {mountain.vertical} m")
print(f"Difficulty: {mountain.difficulty}")
print(f"Beginner friendliness: {mountain.beginner_friendliness}")
print(f"Average icy days: {mountain.average_icy_days}")
print(f"Average snow: {mountain.average_snow}")
print(f"Average rain: {mountain.average_rain}")
print(f"Trail count: {mountain.trail_count()}")
print(f"Lift count: {mountain.lift_count()}")

## Area geometry sampling

Area trails (glades, bowls) store two point sets: `geometry` is the sampled boundary ring, `interior_geometry` is the interior sampling grid used for slope calculations. Plot both for one area trail to sanity-check the sampling.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

area_trail = next(trail for trail in mountain.trails.values() if trail.area)

# geometry/interior_geometry are real shapely objects (Polygon/MultiPoint)
boundary_points = list(area_trail.geometry.exterior.coords)
interior_points = [point.coords[0] for point in area_trail.interior_geometry.geoms]

boundary_lon, boundary_lat, boundary_elev = zip(*boundary_points)
interior_lon, interior_lat, interior_elev = zip(*interior_points)

all_elev = boundary_elev + interior_elev
vmin, vmax = min(all_elev), max(all_elev)
cmap = plt.get_cmap("Blues")

fig, ax = plt.subplots(figsize=(8, 8))
ax.plot(boundary_lon, boundary_lat, color="#c7c7c7", linewidth=1, zorder=1)
points = ax.scatter(
    boundary_lon,
    boundary_lat,
    c=boundary_elev,
    cmap=cmap,
    vmin=vmin,
    vmax=vmax,
    marker="o",
    s=30,
    zorder=2,
)
ax.scatter(
    interior_lon,
    interior_lat,
    c=interior_elev,
    cmap=cmap,
    vmin=vmin,
    vmax=vmax,
    marker="x",
    s=30,
    zorder=2,
)

# color now encodes elevation, so category (boundary vs. interior) is carried by
# marker shape alone -- legend uses neutral gray proxies to show that mapping
legend_handles = [
    Line2D(
        [0],
        [0],
        marker="o",
        linestyle="none",
        markerfacecolor="#888888",
        markeredgecolor="#888888",
        markersize=8,
        label="Boundary (geometry)",
    ),
    Line2D(
        [0],
        [0],
        marker="x",
        linestyle="none",
        color="#888888",
        markersize=8,
        label="Interior grid (interior_geometry)",
    ),
]
ax.legend(handles=legend_handles)

ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.set_title(f"{area_trail.name or area_trail.trail_id} — sampled points by elevation")
ax.set_aspect("equal")

cbar = fig.colorbar(points, ax=ax)
cbar.set_label("Elevation (m)")

plt.show()

## Route finding: least-steep path down

Uses the real `get_area_route()` from `core.support.area_routes` -- the same function `core/osm/osm_processor.py` calls during ingestion -- rather than reimplementing the algorithm here. See that module's docstring for the three-phase approach (bottleneck pass, least-wandering pass, smoothing pass) and its tunable constants (`VERTICAL_BAND_FRACTION`, `START_SLOPE_DEGREES`, etc.).

`get_area_route()` takes a boundary Polygon and interior MultiPoint as geojson blobs (matching what `Trail.geometry`/`Trail.interior_geometry` serialize to) and returns a geojson LineString for the route. Stats about that route (length, vertical drop, steepest segment, steepest sustained pitch) are then read off it with the same `core/support/utils.py` helpers `osm_processor.py` uses for real trails, since a returned route is just as valid a "line" as any other trail's geometry.

In [ ]:
import json

import shapely

from core.support.area_routes import get_area_route
from core.support.utils import get_average_slope, get_length, get_max_slope, get_vertical_drop

boundary_geojson = json.loads(shapely.to_geojson(area_trail.geometry))
interior_geojson = json.loads(shapely.to_geojson(area_trail.interior_geometry))

route = get_area_route(boundary_geojson, interior_geojson)

route_length_m = get_length(route)
route_vertical_m = get_vertical_drop(route)
route_max_slope = get_max_slope(route)
route_average_slope = get_average_slope(route)

print(f"Route: {len(route['coordinates'])} points, {route_length_m:.0f} m long")
print(f"Steepest segment on route: {route_max_slope:.1f} degrees")
print(f"Average segment slope: {route_average_slope:.1f} degrees")
print(f"Vertical drop along route: {route_vertical_m:.0f} m")

In [ ]:
route_lon = [p[0] for p in route["coordinates"]]
route_lat = [p[1] for p in route["coordinates"]]

print(f"Route distance: {route_length_m:.0f} m ({route_length_m * 3.28084:.0f} ft)")

ROUTE_COLOR = "#D62728"  # reads clearly against the Blues elevation ramp

fig, ax = plt.subplots(figsize=(8, 8))
ax.plot(boundary_lon, boundary_lat, color="#c7c7c7", linewidth=1, zorder=1)
points = ax.scatter(
    boundary_lon,
    boundary_lat,
    c=boundary_elev,
    cmap="Blues",
    vmin=vmin,
    vmax=vmax,
    marker="o",
    s=20,
    alpha=0.6,
    zorder=2,
)
ax.scatter(
    interior_lon,
    interior_lat,
    c=interior_elev,
    cmap="Blues",
    vmin=vmin,
    vmax=vmax,
    marker="x",
    s=20,
    alpha=0.6,
    zorder=2,
)
ax.plot(
    route_lon, route_lat, color=ROUTE_COLOR, linewidth=2.5, zorder=3, label="Least-steep route"
)
ax.scatter([route_lon[0]], [route_lat[0]], color=ROUTE_COLOR, marker="^", s=100, zorder=4, label="Top")
ax.scatter(
    [route_lon[-1]], [route_lat[-1]], color=ROUTE_COLOR, marker="v", s=100, zorder=4, label="Bottom"
)

ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.set_title(f"{area_trail.name or area_trail.trail_id} — least-steep route ({route_length_m:.0f} m)")
ax.set_aspect("equal")
ax.legend()

cbar = fig.colorbar(points, ax=ax)
cbar.set_label("Elevation (m)")

plt.show()

## Comparing all areas

Wrap the pattern above (shapely → geojson → `get_area_route()` → stats via `core/support/utils.py`) into a function so it can run on any area trail, then apply it to every area trail at Solitude and compare the routes side by side.

In [ ]:
def analyze_area_route(trail):
    """
    Runs the real get_area_route() pipeline for a single area trail and
    packages it with the boundary/interior points (for plotting) and
    derived stats (via core.support.utils, applied to the route the same
    way osm_processor.py applies them to any other trail's geometry).
    """
    boundary_points = list(trail.geometry.exterior.coords)
    interior_points = [point.coords[0] for point in trail.interior_geometry.geoms]

    boundary_lon, boundary_lat, boundary_elev = zip(*boundary_points)
    interior_lon, interior_lat, interior_elev = zip(*interior_points)

    boundary_geojson = json.loads(shapely.to_geojson(trail.geometry))
    interior_geojson = json.loads(shapely.to_geojson(trail.interior_geometry))

    route = get_area_route(boundary_geojson, interior_geojson)

    return {
        "trail": trail,
        "boundary_lon": boundary_lon,
        "boundary_lat": boundary_lat,
        "boundary_elev": boundary_elev,
        "interior_lon": interior_lon,
        "interior_lat": interior_lat,
        "interior_elev": interior_elev,
        "route": route,
        "route_length_m": get_length(route),
        "vertical_drop_m": get_vertical_drop(route),
        "max_slope": get_max_slope(route),
        "average_slope": get_average_slope(route),
    }


area_trails = [trail for trail in mountain.trails.values() if trail.area]

print(f"{len(area_trails)} area trails found at {mountain.name}")
for i, trail in enumerate(area_trails, start=1):
    print(f"  {i}. {trail.name or trail.trail_id}")

results = [analyze_area_route(trail) for trail in area_trails]

for r in results:
    trail_label = r["trail"].name or r["trail"].trail_id
    print(
        f"\n{trail_label}: {r['route_length_m']:.0f} m route, "
        f"steepest {r['max_slope']:.1f}\N{DEGREE SIGN}, "
        f"{r['vertical_drop_m']:.0f} m vertical"
    )

In [ ]:
import numpy as np


def plot_area_routes(results, title):
    """
    Plots a grid of area-trail maps (small multiples), each showing the
    sampled points colored by elevation and the least-steep route
    overlaid. Reusable across any Mountain's list of analyze_area_route
    results.
    """
    n_results = len(results)
    ncols = min(3, n_results)
    nrows = -(-n_results // ncols)  # ceiling division

    fig, axes = plt.subplots(nrows, ncols, figsize=(6 * ncols, 6 * nrows))
    axes = np.atleast_1d(axes).flatten()
    route_color = "#D62728"  # reads clearly against the Blues elevation ramp

    for ax, r in zip(axes, results):
        trail = r["trail"]
        all_elev = r["boundary_elev"] + r["interior_elev"]
        vmin, vmax = min(all_elev), max(all_elev)

        ax.plot(r["boundary_lon"], r["boundary_lat"], color="#c7c7c7", linewidth=1, zorder=1)
        points = ax.scatter(
            r["boundary_lon"],
            r["boundary_lat"],
            c=r["boundary_elev"],
            cmap="Blues",
            vmin=vmin,
            vmax=vmax,
            marker="o",
            s=15,
            alpha=0.6,
            zorder=2,
        )
        ax.scatter(
            r["interior_lon"],
            r["interior_lat"],
            c=r["interior_elev"],
            cmap="Blues",
            vmin=vmin,
            vmax=vmax,
            marker="x",
            s=15,
            alpha=0.6,
            zorder=2,
        )

        route_lon = [p[0] for p in r["route"]["coordinates"]]
        route_lat = [p[1] for p in r["route"]["coordinates"]]
        ax.plot(route_lon, route_lat, color=route_color, linewidth=2.5, zorder=3, label="Route")
        ax.scatter(
            [route_lon[0]], [route_lat[0]], color=route_color, marker="^", s=100, zorder=4, label="Top"
        )
        ax.scatter(
            [route_lon[-1]], [route_lat[-1]], color=route_color, marker="v", s=100, zorder=4, label="Bottom"
        )

        ax.set_xlabel("Longitude")
        ax.set_ylabel("Latitude")
        ax.set_title(
            f"{trail.name or trail.trail_id}\n"
            f"{r['route_length_m']:.0f} m \N{BULLET OPERATOR} steepest {r['max_slope']:.1f}\N{DEGREE SIGN} "
            f"\N{BULLET OPERATOR} {r['vertical_drop_m']:.0f} m vertical",
            fontsize=10,
        )
        ax.set_aspect("equal")
        ax.legend(fontsize=8)

        fig.colorbar(points, ax=ax, label="Elevation (m)", shrink=0.8)

    for ax in axes[n_results:]:
        ax.axis("off")

    fig.suptitle(title)
    plt.tight_layout()
    plt.show()


plot_area_routes(results, f"Least-steep routes across {mountain.name} area trails")

### Steepest sustained pitches

Reuse `core/support/utils.py`'s `get_steepest_pitch` (the same function used to rate real trails) on each route's point sequence, to see the steepest slope sustained over any 30m/50m/100m stretch -- a different view than the single steepest *segment* printed above, since a route can have one sharp short segment without necessarily having a steep sustained pitch, or vice versa.

In [ ]:
from core.support.utils import get_steepest_pitch

for r in results:
    trail_label = r["trail"].name or r["trail"].trail_id

    steepest_30m = get_steepest_pitch(r["route"], 30)
    steepest_50m = get_steepest_pitch(r["route"], 50)
    steepest_100m = get_steepest_pitch(r["route"], 100)

    print(f"{trail_label}:")
    print(f"  Steepest 30m pitch: {steepest_30m}\N{DEGREE SIGN}")
    print(f"  Steepest 50m pitch: {steepest_50m}\N{DEGREE SIGN}")
    print(f"  Steepest 100m pitch: {steepest_100m}\N{DEGREE SIGN}")

## Another mountain: Crystal Mountain, WA

Sanity-check that the pipeline generalizes: load a different resort's OSM extract and run the same area-route analysis and chart on it.

In [ ]:
CRYSTAL_OSM_FILE = "data/osm/WA/Crystal Mountain.osm"

crystal_season_passes = [Season_Pass.IKON]
crystal_url = "https://www.crystalmountainresort.com"

crystal_mountain = Mountain.from_osm(CRYSTAL_OSM_FILE, crystal_season_passes, crystal_url)
crystal_mountain

In [ ]:
crystal_area_trails = [trail for trail in crystal_mountain.trails.values() if trail.area]

print(f"{len(crystal_area_trails)} area trails found at {crystal_mountain.name}")
for i, trail in enumerate(crystal_area_trails, start=1):
    print(f"  {i}. {trail.name or trail.trail_id}")

crystal_results = [analyze_area_route(trail) for trail in crystal_area_trails]

for r in crystal_results:
    trail_label = r["trail"].name or r["trail"].trail_id
    print(
        f"\n{trail_label}: {r['route_length_m']:.0f} m route, "
        f"steepest {r['max_slope']:.1f}\N{DEGREE SIGN}, "
        f"{r['vertical_drop_m']:.0f} m vertical"
    )

In [ ]:
plot_area_routes(crystal_results, f"Least-steep routes across {crystal_mountain.name} area trails")

### Steepest sustained pitches (Crystal Mountain)

In [ ]:
for r in crystal_results:
    trail_label = r["trail"].name or r["trail"].trail_id

    steepest_30m = get_steepest_pitch(r["route"], 30)
    steepest_50m = get_steepest_pitch(r["route"], 50)
    steepest_100m = get_steepest_pitch(r["route"], 100)

    print(f"{trail_label}:")
    print(f"  Steepest 30m pitch: {steepest_30m}\N{DEGREE SIGN}")
    print(f"  Steepest 50m pitch: {steepest_50m}\N{DEGREE SIGN}")
    print(f"  Steepest 100m pitch: {steepest_100m}\N{DEGREE SIGN}")